# Lab 14: OLS Diagnostics — When the Standard Error Is Lying
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 73 min Core + 20 min Write it yourself + 40 min Extension + take-home Challenge

**Plan on about 1¾ hours** for the required work: setup, Parts 1-4, Write it yourself and the README. The Extension and the Challenge are extra.

---

**Learning Objectives:**
- Read a residual plot and name which assumption each pattern violates
- Show that heteroskedasticity leaves the coefficient alone and breaks the standard error
- Compute robust standard errors and say when they are and are not enough
- Find influential observations with leverage and Cook's distance
- Write a small function that gives the robust standard error for any regression on any slice of the data

**Dataset:** Simulated data with known violations, plus California Housing (scikit-learn, 20,640 districts)

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-4 are GUIDED: run each cell as it is, then answer the questions. "Write it yourself" is required: a short function you write from a blank cell, then three sentences on Part 4 and on what your function found. The Extension is optional, and the Challenge is a take-home.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Write it yourself (required),** after Part 4: `hc3_se()`, a function that fits a regression and gives back the HC3 standard error of one coefficient. About four lines from a blank cell. The check cell below it must print "Passed"; it then runs your function on the districts below the 5.0 ceiling.
> - **Three sentences** after the check, on Part 4's regression and on what your function found, for a reader who will never see your code.
> - The answers to the interpretation questions after Parts 1-3, in text cells.
>
> **You run and read:** everything else. Setup and Parts 1-4 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The Extension (optional) has two one-line blanks in Step 10 (the most influential California districts), simulates firms for you in Step 11, then has two one-line blanks in Step 12 and a short answer.
>
> **The Challenge** (take-home) has no starter code: you write it from the specification, in new cells. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Steps 2 and 4 and the Extension draw random numbers from one generator, `rng`, and each draw moves
it on. So your numbers match the expected output only if you run the cells once, from the top, in
order. If you run a cell twice, use **Runtime → Restart session** and run from the top again.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the random generator
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan   # used in Step 5
from statsmodels.stats.stattools import jarque_bera          # used in Step 5
from sklearn.datasets import fetch_california_housing
from pathlib import Path

rng = np.random.default_rng(seed=42)    # Steps 2 and 4 and the Extension draw from rng, in order

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: The Coefficient Survives, the Interval Does Not (GUIDED — 20 min)

Chapter 14's claim is narrow and important: heteroskedasticity does not bias
the coefficient. It breaks the number you put beside it.

Step 2 simulates 2,000 points on a line with a true slope of 2.0, with errors whose spread grows
with x: the error's standard deviation is 0.5 × x. That is **heteroskedasticity**. Then it fits the
same regression twice. `sm.OLS(y, sm.add_constant(x)).fit()` is Lab 13's regression, with
**classical** standard errors, which assume one error spread for every row. Adding
`cov_type="HC3"` to `.fit()` asks for **robust** (HC3) standard errors instead, which allow the
spread to differ from row to row. `use_t=True` builds the robust interval from the t distribution,
as the classical fit does, so the two intervals differ only in the standard error.

The regression is fitted on plain arrays, so `.params`, `.bse` and `.conf_int()` are read by
position, as in Lab 10's Extension: index 1 is the slope, and `conf_int()[1]` is the slope's
interval, its lower and its upper limit.

In [ ]:
# GUIDED — run as-is
# Step 2: errors whose spread grows with x, fitted with classical and with robust standard errors
n = 2_000
x = rng.uniform(1, 10, n)
TRUE_SLOPE = 2.0
y = 5 + TRUE_SLOPE * x + rng.normal(0, 0.5 * x, n)    # the error's standard deviation is 0.5 * x

fit = sm.OLS(y, sm.add_constant(x)).fit()                              # classical
rob = sm.OLS(y, sm.add_constant(x)).fit(cov_type="HC3", use_t=True)    # robust (HC3)

# <22 pads text to 22 characters; >10.4f right-aligns a number in 10, with 4 decimals
print(f"{'':<22}{'estimate':>10}{'std err':>10}{'95% CI':>22}")
print("-" * 66)
for label, res in [("classical", fit), ("robust (HC3)", rob)]:    # one (label, results) pair per pass
    lo, hi = res.conf_int()[1]
    ci = f"[{lo:.4f}, {hi:.4f}]"
    print(f"{label:<22}{res.params[1]:>10.4f}{res.bse[1]:>10.4f}{ci:>22}")
print(f"\nTrue slope: {TRUE_SLOPE}")

**Expected output:** both rows print the same estimate, 2.0147, against a true slope of 2.0. The
standard error is 0.0264 classical and 0.0290 robust, and the 95% intervals are [1.9629, 2.0664]
and [1.9578, 2.0715].

One sample cannot tell you whether an interval is right. Step 3 asks how often it is: it draws 400
fresh samples of 400 points from the same model, builds both intervals in each, and counts how
many contain the true slope. A correct 95% interval contains it in about 95% of samples. Each
trial gets its own generator, seeded 1000 + i, so Step 3 leaves `rng` untouched.
`lo <= TRUE_SLOPE <= hi` is True when the truth lies inside the interval, and adding True to a
count adds 1. The coverage rate is itself an estimate from 400 trials, so the cell also prints its
**Monte Carlo standard error**, the standard error of a proportion, √(p(1 − p) / 400).

In [ ]:
# GUIDED — run as-is
# Step 3: how often does each interval contain the true slope? 400 simulated samples
TRIALS = 400
covered_classical = 0
covered_robust = 0
for i in range(TRIALS):
    r = np.random.default_rng(1000 + i)          # a fresh generator for each trial
    xi = r.uniform(1, 10, 400)
    yi = 5 + TRUE_SLOPE * xi + r.normal(0, 0.5 * xi, 400)
    Xi = sm.add_constant(xi)

    lo, hi = sm.OLS(yi, Xi).fit().conf_int()[1]
    covered_classical += lo <= TRUE_SLOPE <= hi   # True adds 1, False adds 0
    lo, hi = sm.OLS(yi, Xi).fit(cov_type="HC3", use_t=True).conf_int()[1]
    covered_robust += lo <= TRUE_SLOPE <= hi

# 6.1% prints a share as a percentage, 6 characters wide with 1 decimal: 0.918 prints as 91.8%
for label, count in [("classical", covered_classical), ("robust", covered_robust)]:
    p = count / TRIALS
    mc_se = np.sqrt(p * (1 - p) / TRIALS)
    print(f"{label:>10} interval covers the truth {p:6.1%}  (MC SE {mc_se:.1%})   nominal 95%")

**Expected output:** the classical interval covers the truth in 91.8% of the 400 samples (MC SE
1.4%), and the robust one in 93.5% (MC SE 1.2%), against a nominal 95%.

### Interpretation Questions

1. Both rows of Step 2 fit the same regression to the same data, so only the standard error can differ. Why does heteroskedasticity call for a new standard error but not a new estimate?
2. Compare each coverage rate in Step 3 with the nominal 95%, measuring the gap in units of that rate's Monte Carlo SE. For which of the two intervals is the shortfall clearly more than chance?

*Your answers here:*

1. 
2. 

## Part 2: Reading Residuals (GUIDED — 18 min)

Every assumption behind a standard error is a claim about the residuals, and
the residuals are observable.

Step 4 builds four outcomes on the same 600 values of x. Each is the line 5 + 2x plus noise, with
one thing changed: in the second the noise spreads out as x grows, in the third the true
relationship bends (the −0.25x² term), and in the fourth one row, row 0, has its y set to 60. The
dict `panels` holds the four outcomes under their names, and `.items()` hands back the
(name, outcome) pairs one at a time. For each outcome the cell fits a straight line and plots each
row's residual against its fitted value: `.fittedvalues` holds the fitted values, and `.resid`, as
in Lab 13, the residuals.

In [ ]:
# GUIDED — run as-is
# Step 4: four residual patterns, four different problems
n = 600
xg = rng.uniform(0, 10, n)
panels = {
    "well behaved":      5 + 2 * xg + rng.normal(0, 1.5, n),
    "heteroskedastic":   5 + 2 * xg + rng.normal(0, 0.4 * xg, n),
    "missing curvature": 5 + 2 * xg - 0.25 * xg**2 + rng.normal(0, 1.5, n),
    "an outlier":        5 + 2 * xg + rng.normal(0, 1.5, n),
}
panels["an outlier"][0] = 60        # row 0's outcome, forced far above the line

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
# zip pairs each of the four panels with one (name, outcome) pair
for ax, (name, yv) in zip(axes, panels.items()):
    res = sm.OLS(yv, sm.add_constant(xg)).fit()
    ax.scatter(res.fittedvalues, res.resid, s=6, alpha=0.5, color="darkslategray")
    ax.axhline(0, color="firebrick", lw=1)
    ax.set_title(name)
    ax.set_xlabel("fitted")
axes[0].set_ylabel("residual")
plt.tight_layout()
plt.savefig("figures/ch14_residual_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** four panels, titled with the names in `panels`. In each, ask two things: is
the cloud centred on the red zero line all the way across, and is it the same height all the way
across? Question 1 asks what each departure means.

Step 5 puts a formal test beside each picture. **Breusch-Pagan** regresses the squared residuals on
the regressors and asks whether they explain anything; a small p-value is evidence that the spread
of the errors changes with x. **Jarque-Bera** asks whether the residuals look normal, judged by
their skewness and their tails; a small p-value is evidence that they do not.
`het_breuschpagan(residuals, regressors)` gives back four numbers, and index 1 is the p-value;
`.model.exog` is the table of regressors the fit used, constant included. `jarque_bera(residuals)`
also gives back four numbers, and index 1 is again the p-value.

In [ ]:
# GUIDED — run as-is
# Step 5: a formal test beside each picture
for name, yv in panels.items():
    res = sm.OLS(yv, sm.add_constant(xg)).fit()
    bp = het_breuschpagan(res.resid, res.model.exog)
    jb = jarque_bera(res.resid)
    # <8.4f left-aligns the number in 8 characters, so the second test lines up
    print(f"{name:<20} Breusch-Pagan p={bp[1]:<8.4f} Jarque-Bera p={jb[1]:.4f}")

**Expected output:** four rows of p-values, Breusch-Pagan then Jarque-Bera: well behaved 0.3459
and 0.1955; heteroskedastic 0.0000 and 0.0000; missing curvature 0.7918 and 0.0159; an outlier
0.4590 and 0.0000. A p-value printed as 0.0000 is below 0.00005. A test reports a statistic, not
a verdict: read each row beside its panel in Step 4.

### Interpretation Questions

1. Match each panel to the assumption it violates, and say for each whether it damages the estimate, the standard error, or both.
2. The missing-curvature panel may pass Breusch-Pagan. Explain why a test can miss what a plot shows.

*Your answers here:*

1. 
2. 

## Part 3: Influence (GUIDED — 20 min)

One observation can dominate a regression. *Which* number it dominates is the
interesting part, and it is not always the one you expect.

Step 6 takes Step 4's outlier panel and measures three things for every row: its **leverage**,
its **standardised residual** and its **Cook's distance**. Chapter 14 (§14.6) defines all three.
`.get_influence()` computes them: `.hat_matrix_diag` is each row's leverage,
`.resid_studentized_internal` its standardised residual, and `.cooks_distance` gives back two
arrays, the distances and their p-values, so `[0]` takes the distances. `np.argsort` gives the
positions that would sort an array from smallest to largest, `[::-1]` reverses them, and `[:5]`
keeps the first five, so `order` holds the positions of the five largest, largest first.

In [ ]:
# GUIDED — run as-is
# Step 6: leverage, standardised residual and Cook's distance for the outlier panel
yo = panels["an outlier"]
fit_o = sm.OLS(yo, sm.add_constant(xg)).fit()
infl = fit_o.get_influence()
lev = infl.hat_matrix_diag
std_resid = infl.resid_studentized_internal
cooks = infl.cooks_distance[0]

order = np.argsort(cooks)[::-1][:5]       # the five largest Cook's distances, largest first

print(f"{'obs':>6}{'leverage':>11}{'std resid':>12}{'Cook D':>10}")
print("-" * 40)
for i in order:
    print(f"{i:>6}{lev[i]:>11.4f}{std_resid[i]:>12.3f}{cooks[i]:>10.4f}")

**Expected output:** five rows, the largest Cook's distance first. Row 0, the one Step 4 forced to
60, comes first: leverage 0.0027, standardised residual 18.106, Cook's distance 0.4397. The next,
row 206, has a Cook's distance of 0.0092.

Step 7 deletes row 0 and fits again. `np.delete(array, position)` gives back a copy of the array
without that position; it is applied to both y and x, so the rows still line up.

In [ ]:
# GUIDED — run as-is
# Step 7: delete the most influential row and fit again
top = order[0]                             # the row with the largest Cook's distance
without = sm.OLS(np.delete(yo, top), sm.add_constant(np.delete(xg, top))).fit()

label_with = f"with obs {top}"
label_without = f"without obs {top}"
print(f"{'':<22}{'slope':>10}{'std err':>10}")
print("-" * 42)
print(f"{label_with:<22}{fit_o.params[1]:>10.4f}{fit_o.bse[1]:>10.4f}")
print(f"{label_without:<22}{without.params[1]:>10.4f}{without.bse[1]:>10.4f}")

shift = abs(fit_o.params[1] - without.params[1])
# .0% prints a share as a whole percentage: 0.33 prints as 33%
print(f"\nthe slope moved      {shift:.4f}, which is {shift / fit_o.bse[1]:.2f} of its own standard error")
print(f"the standard error   fell {1 - without.bse[1] / fit_o.bse[1]:.0%}, "
      f"from {fit_o.bse[1]:.4f} to {without.bse[1]:.4f}")

**Expected output:** with row 0, the slope is 2.0325 with a standard error of 0.0317; without it,
2.0142 with 0.0213. The slope moved 0.0182, which is 0.58 of its own standard error, and the
standard error fell 33%, from 0.0317 to 0.0213.

One row out of 600. Note which of the two numbers moved more, and compare that with what happened
in Part 1.

### Interpretation Questions

1. High leverage and a large residual are different things. Define each, and say why Cook's distance combines them.
2. Step 7 printed how far the slope moved and how far its standard error fell when one row went. Which of the two did that row affect more, and why? (Which term of $\hat\sigma^2$ does a single huge residual enter, and with what power?)
3. When is deleting an influential point the right call, and when is it data manipulation?

*Your answers here:*

1. 
2. 
3. 

## Part 4: Real Data (GUIDED — 15 min)

California Housing, where all of this arrives at once.

Step 8 regresses median house value on four district features: median income (`MedInc`), house
age (`HouseAge`), rooms per household (`AveRooms`) and people per household (`AveOccup`). As in
Lab 12, each row is a district, and `value` is in hundreds of thousands of dollars. `fit_ca` is the
classical fit, and `fit_ca_hc3` the same regression with Step 2's HC3 argument. This time the
regression is fitted on a DataFrame, so `.params` and `.bse` are labelled by column name and read
with the name, as Lab 12's Step 7 read the coefficient on `AveRooms`. The loop prints one row per
term, `const` first.

In [ ]:
# GUIDED — run as-is
# Step 8: one real regression, classical and robust standard errors side by side
data = fetch_california_housing(as_frame=True)
ca = data.frame.rename(columns={"MedHouseVal": "value"})    # the outcome gets the shorter name

feat = ["MedInc", "HouseAge", "AveRooms", "AveOccup"]
fit_ca = sm.OLS(ca["value"], sm.add_constant(ca[feat])).fit()
fit_ca_hc3 = sm.OLS(ca["value"], sm.add_constant(ca[feat])).fit(cov_type="HC3", use_t=True)

print(f"{'term':<12}{'coef':>10}{'classical SE':>14}{'HC3 SE':>10}{'ratio':>8}")
print("-" * 54)
for name in fit_ca.params.index:
    ratio = fit_ca_hc3.bse[name] / fit_ca.bse[name]
    print(f"{name:<12}{fit_ca.params[name]:>10.4f}{fit_ca.bse[name]:>14.4f}"
          f"{fit_ca_hc3.bse[name]:>10.4f}{ratio:>8.2f}")

**Expected output:** five rows. For `AveRooms`: coefficient −0.0273, classical SE 0.0024, HC3 SE
0.0121, ratio 5.02. The ratios run from 1.24 (`HouseAge`) to 5.84 (`AveOccup`).

Step 9 draws this regression's residual plot, as Step 4 did, and counts the districts at the
ceiling. The census recorded every district worth more than \$500,000 as 5.00001, so
`ca["value"] > 5` is True for exactly those districts, and `.sum()` counts the Trues. Putting a
comparison like that inside `ca[...]` keeps the rows where it is True, as Lab 8's Step 3 kept the
treated workers with `df[df["treat"] == 1]`: `below` is the table of every district whose value is
at most 5, the ones not at the ceiling. Write it yourself uses it.

In [ ]:
# GUIDED — run as-is
# Step 9: residuals against fitted values, and the districts at the 5.0 ceiling
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(fit_ca.fittedvalues, fit_ca.resid, s=3, alpha=0.15, color="darkslategray")
ax.axhline(0, color="firebrick", lw=1)
ax.set_xlabel("fitted value")
ax.set_ylabel("residual")
ax.set_title("California Housing: residuals vs fitted")
plt.tight_layout()
plt.savefig("figures/ch14_california_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

at_ceiling = (ca["value"] > 5).sum()
below = ca[ca["value"] <= 5]          # every district not at the ceiling
# .2% prints a share as a percentage with 2 decimals; :, adds a thousands comma
print(f"{at_ceiling} of {len(ca):,} districts sit at the ceiling ({at_ceiling / len(ca):.2%})")
print(f"{len(below):,} districts are below it")

**Expected output:** the plot, then "965 of 20,640 districts sit at the ceiling (4.68%)" and
"19,675 districts are below it".

**What to look for:** a straight diagonal edge running down to the right. Those are the districts
at the ceiling: their value is stuck at 5.00001, so the higher the fitted value, the lower their
residual. Then compare the height of the cloud at small and at large fitted values, as you did in
Step 4.

---

## Write it yourself (required — 20 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Step 8 fitted one regression with HC3 standard errors. Step 9 found 965 districts at
the ceiling, whose true value nobody recorded, and made the table `below` without them. Does the
robust standard error on `AveRooms` change when they are left out? Rather than copy Step 8's lines
for a second table, turn them into a function.

Write a function `hc3_se(data, features, term)` that regresses `data["value"]` on the columns
`features` of `data`, with a constant, and gives back the HC3 standard error of the coefficient on
`term`, as one number. With Step 8's table and list, `hc3_se(ca, feat, "AveRooms")` gives the
0.0121 that Step 8 printed.

- It takes about four lines: the `def` line, a one-line docstring, the fit and the `return`.
- Step 8 has the fit you need. Inside the function, use the three arguments wherever Step 8 names
  its table, its list of columns and `AveRooms`.
- Give back one number: not the fitted regression, and not every standard error at once.
- Lab 12's `holdout_rmse()` shows the shape of a function: the `def` line, a one-line docstring,
  the body, and `return`.

Then run the check cell. It calls your function with Step 8's table and compares the answer with
Step 8's; it also tries another term, changed copies of the table and a different list of
features. `assert` stops with its message when its condition is False; the message tells you what
to fix. When everything passes, the check calls `hc3_se(below, feat, "AveRooms")` and prints the
standard error on `AveRooms` with and without the districts at the ceiling.

In [ ]:
# WRITE IT YOURSELF (required): hc3_se(data, features, term)

In [ ]:
# CHECK — for "Write it yourself" above

# Numbers this check compares against, measured from Step 8's regression (HC3 unless named).
# Step 8 printed the first two, rounded, as 0.0121 and 0.0065
SE_AVEROOMS = 0.012123380241071713
SE_MEDINC = 0.00645449638058991
OTHER_ROBUST = {"HC0": 0.010236967836034644, "HC1": 0.010238208004170033, "HC2": 0.011134003076492793}
T_AVEROOMS = -2.255902610578351
P_AVEROOMS = [0.024076731005836703, 0.024087156394603817]    # without and with use_t=True
SE_BELOW = 0.008202902673853516      # AveRooms, in the table below from Step 9


def call(data, features, term):
    """Your hc3_se on these arguments; a plain message instead of a traceback if it stops."""
    # try/except catches an error inside your function, so the message can say where to look
    try:
        return hc3_se(data, features, term)
    except Exception as e:
        hint = "Read the error: it says what went wrong."
        if isinstance(e, KeyError):
            hint = ("A name was not found. features is a list of column names, term is one of them, "
                    "and the outcome column is called value.")
        elif "only integers, slices" in str(e):
            hint = ("Your results hold plain, unlabelled arrays, so .bse cannot be read by name. "
                    "Ask .fit() itself for HC3, as Step 8 does.")
        elif isinstance(e, IndexError):
            hint = "Pick the standard error by name, with term, not by its position."
        elif "broadcast" in str(e):
            hint = "sm.OLS takes the outcome first, then the regressors, as in Step 8."
        elif "not aligned" in str(e):
            hint = "Take the outcome and the columns from the same table, the data argument."
        raise AssertionError(
            f"hc3_se(..., {features}, '{term}') stopped with an error:\n  {e}\n{hint}") from None


def is_rounded(v, full):
    """True if v is full rounded to 2-5 decimals (and not full itself)."""
    # round(v, places) keeps that many decimals
    return any(np.isclose(v, round(full, places), rtol=0, atol=1e-12)
               and not np.isclose(v, full, rtol=0, atol=1e-12) for places in [2, 3, 4, 5])


# globals() holds every name defined so far in this notebook
if "hc3_se" not in globals():
    print("Not written yet: write hc3_se() in the cell above and run it, then run this check.")
else:
    yours = call(ca, feat, "AveRooms")
    assert yours is not None, "hc3_se gave back nothing. Does it end with a return line?"
    assert not hasattr(yours, "bse"), (
        "hc3_se gives back the whole fitted regression. Give back one number: the standard "
        "error on term, read by name.")
    # np.ndim is 0 for a single number
    assert np.ndim(yours) == 0 and isinstance(yours, (int, float, np.floating)), (
        "hc3_se gives back something other than one number: every standard error at once, a list, "
        "or text. Pick out the one for term, by name.")
    assert not np.isclose(yours, fit_ca.bse["AveRooms"]), (
        "That is the classical standard error. Ask .fit() for HC3, as Step 8 does.")
    for kind, se in OTHER_ROBUST.items():
        assert not np.isclose(yours, se), (
            f"That is the {kind} standard error, another version of the robust formula. "
            "This lab uses HC3.")
    assert not np.isclose(yours, fit_ca.params["AveRooms"]), (
        "That is the coefficient on AveRooms. Give back its standard error.")
    assert not np.isclose(yours, SE_AVEROOMS ** 2), (
        "That is the variance. The standard error is its square root, as .bse holds it.")
    assert not np.isclose(yours, T_AVEROOMS) and not np.isclose(yours, fit_ca.tvalues["AveRooms"]), (
        "That is the t statistic. Give back the standard error, from .bse.")
    assert not any(np.isclose(yours, p) for p in P_AVEROOMS + [fit_ca.pvalues["AveRooms"]]), (
        "That is the p-value. Give back the standard error, from .bse.")
    assert not is_rounded(yours, SE_AVEROOMS), (
        "Your number is rounded. Give back the full number; round only when you print.")

    # Changed copies of the table: value moved up by 100, value doubled, and AveRooms doubled
    shifted = ca.copy()
    shifted["value"] = ca["value"] + 100
    value_doubled = ca.copy()
    value_doubled["value"] = 2 * ca["value"]
    rooms_doubled = ca.copy()
    rooms_doubled["AveRooms"] = 2 * ca["AveRooms"]
    assert np.isclose(call(shifted, feat, "AveRooms"), yours), (
        "Adding 100 to every value changes your standard error, so the regression has no "
        "intercept. Add the column of 1s with sm.add_constant, as Step 8 does.")
    assert np.isclose(call(value_doubled, feat, "AveRooms"), 2 * yours), (
        "Doubling every value in the table you pass in should double the standard error, and "
        "yours did not change that way. Does the function take value from its data argument, "
        "or from the notebook's ca?")
    assert not np.isclose(call(rooms_doubled, feat, "AveRooms"), yours), (
        "Doubling AveRooms in the table you pass in leaves your answer unchanged. Does the "
        "function take its columns from its data argument, or from the notebook's ca?")
    assert not np.isclose(call(ca, ["MedInc", "AveRooms"], "AveRooms"), yours), (
        "hc3_se(ca, ['MedInc', 'AveRooms'], 'AveRooms') gives the same answer as all four features. "
        "Does the function use its features argument, or the notebook's feat?")
    medinc = call(ca, feat, "MedInc")
    assert not np.isclose(medinc, yours), (
        "hc3_se(ca, feat, 'MedInc') gives the AveRooms answer. Use the term argument to pick the "
        "standard error.")
    assert np.isclose(medinc, SE_MEDINC), (
        f"hc3_se(ca, feat, 'MedInc') gives {medinc:.4f}, and Step 8's HC3 column printed 0.0065.")
    assert np.isclose(yours, SE_AVEROOMS), (
        f"hc3_se(ca, feat, 'AveRooms') gives {yours:.4f}, and Step 8's HC3 column printed 0.0121. "
        "Compare your function's fit with Step 8's, line by line.")
    se_below = call(below, feat, "AveRooms")
    assert np.isclose(se_below, SE_BELOW), (
        f"On the table below, hc3_se gives {se_below:.4f} for AveRooms, but it should give "
        f"{SE_BELOW:.4f}. Does the function use only its arguments?")
    print("Passed. hc3_se matches Step 8, and uses the table, the features and the term it is given.")
    print(f"HC3 standard error on AveRooms: all districts {yours:.4f}, below the ceiling {se_below:.4f}")

### Your answer: Part 4 and Write it yourself

Three sentences, for a reader who will never see your code:

1. Report the coefficient on `AveRooms` with the standard error you would use, and say why that one.
2. Say what you would tell the reader about the 965 districts at the ceiling.
3. The check printed the HC3 standard error on `AveRooms` with and without those districts. Would a smaller standard error be a reason to leave them out? (Part 3's last question is a guide.)

*Your answers here:*

1. 
2. 
3. 

---

## Extension (Optional — 40 min): Influence in Real Data, and When Robust Is Not Enough

Two optional pieces. Step 10 finds the California districts that Step 8's regression leans on
hardest. Steps 11 and 12 show a problem that robust standard errors cannot fix. If you skip the
Extension, leave Steps 10 and 12 unrun, so your submitted notebook shows no error.

### The most influential districts (Step 10)

Step 10 measures influence in Step 8's regression, as Step 6 did in the simulated one. Your two
blanks are Cook's distance for every district and the positions of the five largest. The cell then
prints those five districts: `.iloc[worst]` picks rows by position, as Lab 13's `.iloc[i]` did, and
the new column `cooks_d` is each one's Cook's distance. The last line asks how much of the total
Cook's distance, summed over all 20,640 districts, those five carry.

In [ ]:
# EXTENSION (optional) — fill in the two blanks (____)
# Step 10: the five most influential districts
# Blank 1: Cook's distance for every district, from fit_ca, as in Step 6
cooks_ca = ____
# Blank 2: the positions of the five largest, largest first, as in Step 6
worst = ____

top5 = ca.iloc[worst][feat + ["value"]].copy()     # a copy, so the new column leaves ca alone
top5["cooks_d"] = cooks_ca[worst]
print(top5)

share = cooks_ca[worst].sum() / cooks_ca.sum()
print(f"\n5 of {len(ca):,} districts ({5 / len(ca):.3%}) carry {share:.0%} of the total Cook's distance")

**Expected output:** five districts, the largest Cook's distance first: row 19006 (6.33), with
1,243 people per household, then row 1914 (4.25), with 141.9 rooms per household and a value of
5.00001. Together the five, 0.024% of the districts, carry 92% of the total Cook's distance. If the
first two rows are 0 and 206, you reused Step 6's simulated results: with a share of 52%, Blank 1
used `infl` or `cooks`; with 0%, Blank 2 used `order`. If the share is 0% and row 0 is not first,
you took the five smallest. If the share is 92% but row 19006 is not first, check the order of your
five against Step 6's. *ValueError: Buffer has wrong number of dimensions* means Blank 1 kept both
arrays that `.cooks_distance` gives back.

The second cell of Step 10 fits the regression again without those five districts.
`ca.index[worst]` turns their positions into row labels, and `.drop()` gives back a copy of `ca`
without those rows.

In [ ]:
# EXTENSION — run as-is
# Step 10, continued: fit again without the five, and compare the coefficients
trimmed = ca.drop(ca.index[worst])
refit = sm.OLS(trimmed["value"], sm.add_constant(trimmed[feat])).fit()

print(f"{'term':<12}{'with all':>12}{'without 5':>12}{'change':>10}")
for name in fit_ca.params.index:
    change = refit.params[name] - fit_ca.params[name]
    # +10.4f always prints the sign
    print(f"{name:<12}{fit_ca.params[name]:>12.4f}{refit.params[name]:>12.4f}{change:>+10.4f}")

**Expected output:** five rows. `AveRooms` goes from −0.0273 to −0.0492, `AveOccup` from −0.0045 to
−0.0130, and `MedInc` from 0.4433 to 0.4522.

### Clustered errors (Steps 11 and 12)

Robust standard errors handle heteroskedasticity. They do nothing about dependence between
observations.

Step 11 is done for you. It simulates 40 firms with 25 employees each, 1,000 workers in all, with a
true slope of 1.5. Each worker's outcome includes a shock that every worker in the same firm
shares, so two workers in one firm have correlated errors. `np.repeat(np.arange(40), 25)` gives
each worker a firm number: 0 twenty-five times, then 1 twenty-five times, and so on.
`rng.normal(0, 2.0, 40)[firm]` draws one shock per firm and hands each worker their own firm's.

In [ ]:
# EXTENSION — run as-is
# Step 11: 40 firms of 25 employees, with a shock shared inside each firm
n_firms = 40
per_firm = 25
TRUE_B = 1.5
firm = np.repeat(np.arange(n_firms), per_firm)        # each worker's firm: 0, 0, ..., 1, 1, ..., 39
firm_shock = rng.normal(0, 2.0, n_firms)[firm]         # one shock per firm, copied to its workers
xc = rng.normal(0, 1, n_firms * per_firm)
yc = 1.0 + TRUE_B * xc + firm_shock + rng.normal(0, 1.0, n_firms * per_firm)

print(f"{len(yc):,} workers in {n_firms} firms; true slope {TRUE_B}")

**Expected output:** 1,000 workers in 40 firms; true slope 1.5.

Step 12 fits the same regression three ways. **Clustered** standard errors allow the errors of
rows in the same group to be correlated, and they take one more argument than HC3:
`.fit(cov_type="cluster", cov_kwds={"groups": labels}, use_t=True)`, where `labels` gives each
row's group. `cov_kwds` is a dict of settings for that kind of standard error. Your two blanks are
the HC3 fit and the fit clustered by firm.

In [ ]:
# EXTENSION (optional) — fill in the two blanks (____)
# Step 12: classical, robust and clustered standard errors for the same slope
Xc = sm.add_constant(xc)
ext_classical = sm.OLS(yc, Xc).fit()
# Blank 1: HC3 standard errors, as in Step 2
ext_hc3 = ____
# Blank 2: standard errors clustered by firm
ext_cluster = ____

print(f"true slope {TRUE_B}, estimate {ext_classical.params[1]:.3f}\n")
print(f"{'errors':<18}{'SE':>10}{'t':>9}{'95% CI':>26}")
print("-" * 63)
for label, res in [("classical", ext_classical), ("HC3 robust", ext_hc3), ("cluster by firm", ext_cluster)]:
    lo, hi = res.conf_int()[1]
    print(f"{label:<18}{res.bse[1]:>10.4f}{res.params[1] / res.bse[1]:>9.2f}     [{lo:>7.3f}, {hi:>7.3f}]")
print(f"\ncluster SE / classical SE = {ext_cluster.bse[1] / ext_classical.bse[1]:.2f}x")
print(f"cluster SE / HC3 SE       = {ext_cluster.bse[1] / ext_hc3.bse[1]:.2f}x")

**Expected output:** an estimate of 1.521, then three rows. Standard errors 0.0812 (classical),
0.0857 (HC3) and 0.0948 (clustered by firm), with 95% intervals [1.362, 1.681], [1.353, 1.689] and
[1.329, 1.713]. The clustered standard error is 1.17 times the classical one and 1.11 times HC3's.
If the clustered interval reads [1.335, 1.707], the cluster fit is missing `use_t=True`. An HC3
standard error of 0.0854 is HC1's, a different version of the robust formula.

**Your answer.** Rank the three standard errors by size and explain the ordering. Then say what
HC3 assumed that was false here.

*Your answer here:*

---

## Challenge (Take-Home): The Specification That Passes Everything

Build a regression that passes every diagnostic in this lab and is still wrong. There is no
starter code: write it in the cell below and in new cells under it.

1. Simulate data where y depends on x AND on a confounder u: a variable that also drives x.
2. Regress y on x alone. Confirm that the residuals pass Step 5's two tests and that no row has a
   Cook's distance above 1, a common rule for a single row that dominates the fit (Step 6 shows how
   to compute it). The stricter 4/n screen flags some rows in almost any regression: it says where
   to look, not that something is wrong.
3. Report the coefficient against the truth.
4. Reflect: diagnostics test the model's *internal* assumptions. Name what they cannot test, and
   which chapter supplies that tool.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated errors whose spread grows with x, and fitted one regression
  with classical and with robust (HC3) standard errors: the same slope,
  [YOUR VALUE], with standard errors of [YOUR VALUE] and [YOUR VALUE]
* Ran 400 simulated samples: the classical 95% interval covered the true
  slope in [YOUR VALUE] of them and the robust one in [YOUR VALUE], with
  Monte Carlo standard errors of [YOUR VALUE] and [YOUR VALUE]
* Read four residual plots beside Breusch-Pagan and Jarque-Bera tests
* Deleted one row out of 600: the slope moved [YOUR VALUE] of its own
  standard error, and the standard error fell [YOUR VALUE]
* Fitted a California Housing regression: the HC3 standard error on
  AveRooms was [YOUR VALUE] times the classical one
* Wrote a function, hc3_se(): the HC3 standard error on AveRooms was
  [YOUR VALUE] with all districts and [YOUR VALUE] without the 965 at
  the 5.0 ceiling
* (Only if I did the Extension) Five districts carried [YOUR VALUE] of
  the total Cook's distance
* (Only if I did the take-home Challenge) A regression that passed
  every diagnostic and was still wrong: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: OLS Diagnostics: When the Standard Error Is Lying
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'cutting-edge'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab14-ols-diagnostics`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab14-ols-diagnostics`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab14-ols-diagnostics`, branch `main`,
   commit message `Lab 14 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.